# Phase 2 — Dataset exploration (EmoDJ)

## Objective
Understand the **music recommendation dataset** before writing the algorithm.
This notebook does **not** train an emotion classifier and does **not** recommend songs yet.

We answer:
1. How many songs do we have, and are they balanced across 6 emotions × 2 categories?
2. Are there missing values, duplicate tracks, or weak YouTube links?
3. What does that imply for `recommend_songs()` in Phase 3?

## Why Jupyter here?
Exploration is iterative (tables + charts). Production logic will later live in `.py` modules so Flutter and tests do not depend on this notebook.

## How to run
From `recommendation_system/`:

```text
pip install -r requirements.txt
jupyter notebook notebooks/01_dataset_exploration.ipynb
```

Figures are also written to `notebooks/figures/` so they can be used in a report without re-running the kernel.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.titlesize"] = 12

NOTEBOOK_DIR = Path.cwd()
CANDIDATES = [
    NOTEBOOK_DIR / "dataset" / "songs.csv",
    NOTEBOOK_DIR / "recommendation_system" / "dataset" / "songs.csv",
    NOTEBOOK_DIR.parent / "dataset" / "songs.csv",
    NOTEBOOK_DIR.parent.parent / "recommendation_system" / "dataset" / "songs.csv",
]
CSV_PATH = next((p.resolve() for p in CANDIDATES if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        "Could not find songs.csv. Expected recommendation_system/dataset/songs.csv."
    )

FIGURES_DIR = CSV_PATH.parent.parent / "notebooks" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def show_and_save(fig, name: str) -> None:
    path = FIGURES_DIR / name
    fig.savefig(path, dpi=140, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print(f"Saved {path}")

print(f"Loaded catalog from: {CSV_PATH}")
print(f"Figures directory:   {FIGURES_DIR}")

Loaded catalog from: C:\Users\yameo\Downloads\EmoDJ\recommendation_system\dataset\songs.csv
Figures directory:   C:\Users\yameo\Downloads\EmoDJ\recommendation_system\notebooks\figures


## 1. Load and inspect the schema

Expected columns (Phase 1 contract):
- `emotion` — lowercase: sadness, joy, love, anger, fear, surprise
- `category` — lowercase: classic or rap
- `title`, `artist`, `youtube_url`

In [2]:
df = pd.read_csv(CSV_PATH)
print("Shape (rows, columns):", df.shape)
print("Columns:", list(df.columns))
print("\nDtypes:")
print(df.dtypes)
df.head(8)

Shape (rows, columns): (60, 5)
Columns: ['emotion', 'category', 'title', 'artist', 'youtube_url']

Dtypes:
emotion        str
category       str
title          str
artist         str
youtube_url    str
dtype: object


,emotion,category,title,artist,youtube_url
0,sadness,classic,Someone Like You,Adele,https://www.youtube.com/watch?v=hLQl3WQQoQ0
1,sadness,classic,Someone You Loved,Lewis Capaldi,https://www.youtube.com/watch?v=zABLecsR5UE
2,sadness,classic,when the party's over,Billie Eilish,https://www.youtube.com/watch?v=pbMwTqkKSps
3,sadness,classic,Stay With Me,Sam Smith,https://www.youtube.com/watch?v=pB-5XG-DbAA
4,sadness,classic,Somebody That I Used to Know,Gotye,https://www.youtube.com/watch?v=8UVNT4wvIGY
5,joy,classic,Happy,Pharrell Williams,https://www.youtube.com/watch?v=y6Sxv-sUYtM
6,joy,classic,CAN'T STOP THE FEELING!,Justin Timberlake,https://www.youtube.com/watch?v=ru0K8uYEZWw
7,joy,classic,I Wanna Dance with Somebody,Whitney Houston,https://www.youtube.com/watch?v=eH3giaIzONA


## 2. Missing values

A recommendation row is unusable without emotion, category, title, or artist.
A missing YouTube URL can be replaced later by a search link, but we still flag it.

In [3]:
missing = df.isna().sum().to_frame("missing_count")
missing["missing_pct"] = (missing["missing_count"] / len(df) * 100).round(1)
empty_strings = (df.astype(str).apply(lambda s: s.str.strip() == "") | (df.astype(str) == "nan")).sum()
missing["empty_or_blank"] = empty_strings
missing

,missing_count,missing_pct,empty_or_blank
emotion,0,0.0,0
category,0,0.0,0
title,0,0.0,0
artist,0,0.0,0
youtube_url,0,0.0,0


## 3. Distribution: 6 emotions × 2 categories

The prototype target is **at least 5 songs per (emotion, category) bin**, so **60 songs total**.

In [4]:
EXPECTED_EMOTIONS = ["sadness", "joy", "love", "anger", "fear", "surprise"]
EXPECTED_CATEGORIES = ["classic", "rap"]
REQUIRED_COLUMNS = ["emotion", "category", "title", "artist", "youtube_url"]

df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()
df["category"] = df["category"].astype(str).str.strip().str.lower()

print("Unique emotions:", sorted(df["emotion"].unique()))
print("Unique categories:", sorted(df["category"].unique()))
print("Unexpected emotions:", sorted(set(df["emotion"]) - set(EXPECTED_EMOTIONS)))
print("Unexpected categories:", sorted(set(df["category"]) - set(EXPECTED_CATEGORIES)))

counts = (
    df.pivot_table(index="emotion", columns="category", values="title", aggfunc="count")
    .reindex(index=EXPECTED_EMOTIONS, columns=EXPECTED_CATEGORIES)
    .fillna(0)
    .astype(int)
)
counts["total"] = counts.sum(axis=1)
counts.loc["total"] = counts.sum(axis=0)
counts

Unique emotions: ['anger', 'fear', 'joy', 'love', 'sadness', 'surprise']
Unique categories: ['classic', 'rap']
Unexpected emotions: []
Unexpected categories: []


category,classic,rap,total
emotion,,,
sadness,5,5,10
joy,5,5,10
love,5,5,10
anger,5,5,10
fear,5,5,10
surprise,5,5,10
total,30,30,60


In [5]:
bin_counts = (
    df.groupby(["emotion", "category"], dropna=False)
    .size()
    .reindex(
        pd.MultiIndex.from_product(
            [EXPECTED_EMOTIONS, EXPECTED_CATEGORIES], names=["emotion", "category"]
        ),
        fill_value=0,
    )
)
thin_bins = bin_counts[bin_counts < 5]
print("Bins with fewer than 5 songs:")
if thin_bins.empty:
    print("None — every emotion × category has at least 5 tracks.")
else:
    print(thin_bins)

Bins with fewer than 5 songs:
None — every emotion × category has at least 5 tracks.


## 4. Visualizations

Charts should match the count table above. If they do not, the plot is wrong — not the CSV.

In [6]:
plot_df = df[
    df["emotion"].isin(EXPECTED_EMOTIONS) & df["category"].isin(EXPECTED_CATEGORIES)
].copy()
plot_df["emotion"] = pd.Categorical(plot_df["emotion"], EXPECTED_EMOTIONS, ordered=True)
plot_df["category"] = pd.Categorical(plot_df["category"], EXPECTED_CATEGORIES, ordered=True)

fig, ax = plt.subplots()
sns.countplot(data=plot_df, x="emotion", hue="category", ax=ax, palette="Set2")
ax.set_title("Songs per emotion and category")
ax.set_xlabel("Emotion (classifier labels)")
ax.set_ylabel("Number of songs")
ax.legend(title="Category")
fig.tight_layout()
show_and_save(fig, "songs_by_emotion_category.png")

<Figure size 900x450 with 1 Axes>

Saved C:\Users\yameo\Downloads\EmoDJ\recommendation_system\notebooks\figures\songs_by_emotion_category.png


In [7]:
heat = (
    plot_df.pivot_table(index="emotion", columns="category", values="title", aggfunc="count")
    .reindex(index=EXPECTED_EMOTIONS, columns=EXPECTED_CATEGORIES)
    .fillna(0)
    .astype(int)
)

fig, ax = plt.subplots(figsize=(5.5, 5))
sns.heatmap(heat, annot=True, fmt="d", cmap="YlGnBu", cbar=False, ax=ax)
ax.set_title("Catalog heatmap (emotion × category)")
ax.set_xlabel("Category")
ax.set_ylabel("Emotion")
fig.tight_layout()
show_and_save(fig, "heatmap_emotion_category.png")

<Figure size 550x500 with 1 Axes>

Saved C:\Users\yameo\Downloads\EmoDJ\recommendation_system\notebooks\figures\heatmap_emotion_category.png


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_df["emotion"].value_counts().reindex(EXPECTED_EMOTIONS).plot(
    kind="bar", ax=axes[0], color="#4C78A8", rot=30
)
axes[0].set_title("Songs per emotion (classic + rap)")
axes[0].set_ylabel("Count")

plot_df["category"].value_counts().reindex(EXPECTED_CATEGORIES).plot(
    kind="bar", ax=axes[1], color=["#54A24B", "#E45756"], rot=0
)
axes[1].set_title("Songs per category (all emotions)")
axes[1].set_ylabel("Count")
fig.tight_layout()
show_and_save(fig, "totals_emotion_and_category.png")

<Figure size 1000x400 with 2 Axes>

Saved C:\Users\yameo\Downloads\EmoDJ\recommendation_system\notebooks\figures\totals_emotion_and_category.png


## 5. Duplicates and repeated artists

The algorithm will later deduplicate on `(title, artist)` so the same card is not shown twice.
Repeated artists across different songs are allowed (e.g. Adele in sadness and anger).

In [9]:
key = df[["title", "artist"]].apply(lambda s: s.astype(str).str.strip().str.lower())
dup_mask = key.duplicated(keep=False)
print(f"Duplicate (title, artist) rows: {int(dup_mask.sum())}")
if dup_mask.any():
    display(df.loc[dup_mask, ["emotion", "category", "title", "artist"]])

url_dups = df["youtube_url"].astype(str).str.strip().duplicated(keep=False)
print(f"Duplicate youtube_url rows: {int(url_dups.sum())}")
if url_dups.any():
    display(df.loc[url_dups, ["title", "artist", "youtube_url"]])

print("\nArtists appearing more than once:")
artist_counts = df["artist"].value_counts()
repeat_artists = artist_counts[artist_counts > 1].to_frame("song_count")
display(repeat_artists)

Duplicate (title, artist) rows: 0
Duplicate youtube_url rows: 0

Artists appearing more than once:


,song_count
artist,
Eminem,4
Kanye West,3
Adele,2
Whitney Houston,2
Kendrick Lamar,2


## 6. YouTube URL quality

Two accepted forms from Phase 1:
- **watch URL** — `https://www.youtube.com/watch?v=...` (preferred, specific video)
- **search URL** — `https://www.youtube.com/results?search_query=...` (fallback when a stable official ID was not used)

This cell classifies URLs. It does **not** call the YouTube API.

In [10]:
urls = df["youtube_url"].astype(str).str.strip()

def url_kind(u: str) -> str:
    if not u or u.lower() in {"nan", "none"}:
        return "missing"
    if "youtube.com/watch?v=" in u or "youtu.be/" in u:
        return "watch"
    if "youtube.com/results?search_query=" in u:
        return "search"
    return "other"

df["url_kind"] = urls.map(url_kind)
kind_counts = df["url_kind"].value_counts()
print(kind_counts.to_string())
print(f"\nWatch URLs: {kind_counts.get('watch', 0)} / {len(df)}")
print(f"Search fallbacks: {kind_counts.get('search', 0)} / {len(df)}")

fig, ax = plt.subplots(figsize=(5, 3.5))
kind_counts.plot(kind="bar", ax=ax, color="#72B7B2", rot=0)
ax.set_title("YouTube URL types in the catalog")
ax.set_ylabel("Count")
fig.tight_layout()
show_and_save(fig, "youtube_url_types.png")

search_rows = df.loc[
    df["url_kind"] == "search", ["emotion", "category", "title", "artist", "youtube_url"]
]
print("\nRows using a YouTube search URL (not a specific video id):")
search_rows

url_kind
watch     52
search     8

Watch URLs: 52 / 60
Search fallbacks: 8 / 60


<Figure size 500x350 with 1 Axes>

Saved C:\Users\yameo\Downloads\EmoDJ\recommendation_system\notebooks\figures\youtube_url_types.png

Rows using a YouTube search URL (not a specific video id):


,emotion,category,title,artist,youtube_url
42,love,rap,The Light,Common,https://www.youtube.com/results?search_query=C...
43,love,rap,You're All I Need,Method Man,https://www.youtube.com/results?search_query=M...
44,love,rap,Love Yourz,J. Cole,https://www.youtube.com/results?search_query=J...
48,anger,rap,Party Up,DMX,https://www.youtube.com/results?search_query=D...
49,anger,rap,Black Skinhead,Kanye West,https://www.youtube.com/results?search_query=K...
51,fear,rap,Keep Ya Head Up,2Pac,https://www.youtube.com/results?search_query=2...
52,fear,rap,Blessings,Chance the Rapper,https://www.youtube.com/results?search_query=C...
54,fear,rap,Ultralight Beam,Kanye West,https://www.youtube.com/results?search_query=K...


## 7. Sample rows per bin

A quick visual check that Classic and Rap catalogs are distinct, and that `fear` songs look like comforting tracks (curator intent), not horror/scare music.

In [11]:
preview_cols = ["title", "artist"]
for emotion in EXPECTED_EMOTIONS:
    print("=" * 72)
    print(emotion.upper())
    for category in EXPECTED_CATEGORIES:
        subset = df[(df["emotion"] == emotion) & (df["category"] == category)][preview_cols]
        print(f"\n  [{category}] {len(subset)} songs")
        print(subset.to_string(index=False))
        print()

SADNESS

  [classic] 5 songs
                       title        artist
            Someone Like You         Adele
           Someone You Loved Lewis Capaldi
       when the party's over Billie Eilish
                Stay With Me     Sam Smith
Somebody That I Used to Know         Gotye


  [rap] 5 songs
         title       artist
  Lucid Dreams   Juice WRLD
          SAD! XXXTENTACION
  Let You Down           NF
1-800-273-8255        Logic
   Mockingbird       Eminem

JOY

  [classic] 5 songs
                      title                artist
                      Happy     Pharrell Williams
    CAN'T STOP THE FEELING!     Justin Timberlake
I Wanna Dance with Somebody       Whitney Houston
                Uptown Funk           Mark Ronson
        Walking on Sunshine Katrina and the Waves


  [rap] 5 songs
               title                  artist
             Hey Ya!                 OutKast
Young, Wild and Free              Snoop Dogg
       Can't Hold Us Macklemore & Ryan Lewis
   

## 8. Interpretation for the recommendation algorithm

### What the numbers mean
- If every heatmap cell is **5**, Stay-in-my-mood with `limit=5` can fill a full list **without mixing categories or emotions**.
- Change-my-mood will read a **different emotion column** (mapping decided in Phase 0). Those target bins are the same size, so Mode B is not starved of data.
- Classic and Rap each hold half the catalog. A category filter is a hard cut: the user never receives the other genre from this CSV.

### Limitations visible in the data (state these in the report)
1. **Small bins** — 5 songs per cell means repeats if the user asks several times, unless we shuffle.
2. **Curator labels, not audio features** — Adele can appear under sadness and anger; the tag is a human judgment, not a model output.
3. **Fear bin** — tracks such as *Fix You* or *Lean on Me* are soothing. The label matches the **user's detected emotion**, not the song's own affect. Document this so Mode A for fear is not misunderstood as "scary music".
4. **Search URLs** — some rap rows open a YouTube search instead of one official video. The UI should still work; the experience is slightly less precise.
5. **Artist concentration** — a few artists appear more than once. That is acceptable, but the catalog is not a broad survey of each genre.

### What we will **not** do in Phase 3 because of this EDA
- We will not mix Classic and Rap to "fill" a short list.
- We will not silently steal songs from another emotion if a bin is short; we return fewer songs (unless you later approve a fallback for Change-my-mood only).
- We will not generate fake titles at runtime; the CSV is the only catalog.

## 9. Phase 2 validation checklist

Re-run this notebook after any CSV edit. The code cell below prints PASS/FAIL so the report stays honest.

In [12]:
checks = []

def check(name: str, ok: bool, detail: str) -> None:
    checks.append((name, ok, detail))
    status = "PASS" if ok else "FAIL"
    print(f"[{status}] {name}: {detail}")

check("row_count_ge_60", len(df) >= 60, f"{len(df)} rows")
check(
    "required_columns",
    set(REQUIRED_COLUMNS).issubset(df.columns),
    str(REQUIRED_COLUMNS),
)
check(
    "no_missing_core_fields",
    df[["emotion", "category", "title", "artist"]].isna().sum().sum() == 0,
    "core fields",
)
check(
    "emotions_only_expected",
    set(df["emotion"]).issubset(EXPECTED_EMOTIONS),
    str(sorted(df["emotion"].unique())),
)
check(
    "categories_only_expected",
    set(df["category"]).issubset(EXPECTED_CATEGORIES),
    str(sorted(df["category"].unique())),
)
check("min_5_per_bin", bool((bin_counts >= 5).all()), f"min bin size = {int(bin_counts.min())}")
check("no_title_artist_duplicates", not key.duplicated().any(), "(title, artist) uniqueness")
check(
    "all_urls_present",
    int(df["url_kind"].eq("missing").sum()) == 0,
    f"missing={int(df['url_kind'].eq('missing').sum())}",
)
check(
    "urls_are_watch_or_search",
    set(df["url_kind"]).issubset({"watch", "search"}),
    str(df["url_kind"].value_counts().to_dict()),
)

summary = pd.DataFrame(checks, columns=["check", "passed", "detail"])
print(f"\n{int(summary['passed'].sum())} / {len(summary)} checks passed")
summary

[PASS] row_count_ge_60: 60 rows
[PASS] required_columns: ['emotion', 'category', 'title', 'artist', 'youtube_url']
[PASS] no_missing_core_fields: core fields
[PASS] emotions_only_expected: ['anger', 'fear', 'joy', 'love', 'sadness', 'surprise']
[PASS] categories_only_expected: ['classic', 'rap']
[PASS] min_5_per_bin: min bin size = 5
[PASS] no_title_artist_duplicates: (title, artist) uniqueness
[PASS] all_urls_present: missing=0
[PASS] urls_are_watch_or_search: {'watch': 52, 'search': 8}

9 / 9 checks passed


,check,passed,detail
0,row_count_ge_60,True,60 rows
1,required_columns,True,"['emotion', 'category', 'title', 'artist', 'yo..."
2,no_missing_core_fields,True,core fields
3,emotions_only_expected,True,"['anger', 'fear', 'joy', 'love', 'sadness', 's..."
4,categories_only_expected,True,"['classic', 'rap']"
5,min_5_per_bin,True,min bin size = 5
6,no_title_artist_duplicates,True,"(title, artist) uniqueness"
7,all_urls_present,True,missing=0
8,urls_are_watch_or_search,True,"{'watch': 52, 'search': 8}"
